# FinSmart — FinBot & Sistem Rekomendasi Anggaran
**Coding Camp 2026 | CC26-PSU407 | Muhammad Syaiful**

---

## Isi Notebook

| No | Bagian | Section |
|----|--------|---------|
| 1 | Install dan Import Library | Section 1 |
| 2 | Load Dataset | Section 2 |
| 3 | FinBot — Chatbot Keuangan (Hugging Face) | Section 3-5 |
| 4 | Sistem Rekomendasi Anggaran Personal | Section 6-7 |
| 5 | Dokumentasi Arsitektur Pipeline AI | Section 8 |
| 6 | FastAPI Lengkap (Semua Endpoint) | Section 9 |

> Notebook ini dijalankan setelah `FinSmart_AI_Engineer_Final.ipynb` selesai,
> karena membutuhkan file `encoders.pkl`, `scaler.pkl`, dan `xgb_model.pkl`.

---
## 1. Install dan Import Library

In [1]:
!pip install transformers torch scikit-learn fastapi uvicorn -q

print("Instalasi selesai")

Instalasi selesai


In [2]:
import numpy as np
import pandas as pd
import json
import os
import pickle
from transformers import pipeline

print("Semua library berhasil diimport")

Semua library berhasil diimport


---
## 2. Load Dataset

In [3]:
df = pd.read_csv("personal_finance_dataset_8000_extended.csv", sep=";")

print(f"Shape dataset : {df.shape}")
print(f"Kolom         : {list(df.columns)}")
df.head(3)

Shape dataset : (8000, 15)
Kolom         : ['Date', 'Description', 'Amount', 'Category', 'PaymentMethod', 'Location', 'AccountType', 'TransactionType', 'DeviceUsed', 'Currency', 'MerchantType', 'LoyaltyProgram', 'Weekday', 'Month', 'TimeOfDay']


,Date,Description,Amount,Category,PaymentMethod,Location,AccountType,TransactionType,DeviceUsed,Currency,MerchantType,LoyaltyProgram,Weekday,Month,TimeOfDay
0,05/04/2025,Transaction at IRCTC,2066649,Travel,Debit Card,Kolkata,Salary,Debit,Mobile,INR,Service,No,Saturday,April,Night
1,06/12/2025,Transaction at Myntra,191905,Online Shopping,Net Banking,Hyderabad,Salary,Debit,Desktop,INR,Online Store,Yes,Saturday,December,Evening
2,20/06/2025,Transaction at BigBazaar,283768,Grocery,Debit Card,Bangalore,Savings,Debit,Mobile,INR,Retail,No,Friday,June,Evening


---
## 3. FinBot — Definisi Intent dan Database Respons

FinBot menggunakan pendekatan **Zero-Shot Classification** dari Hugging Face untuk mendeteksi intent dari pertanyaan pengguna, kemudian memetakannya ke respons yang telah didefinisikan.

Model yang digunakan: `facebook/bart-large-mnli`

In [4]:
import warnings
warnings.filterwarnings("ignore")

print("Loading model Hugging Face, harap tunggu...")

finbot_classifier = pipeline(
    task="zero-shot-classification",
    model="facebook/bart-large-mnli"
)

print("Model FinBot berhasil diload")

Loading model Hugging Face, harap tunggu...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Model FinBot berhasil diload


In [5]:
INTENT_LABELS = [
    "tips menabung",
    "cara mengatur anggaran",
    "kategori pengeluaran",
    "investasi untuk pemula",
    "cara mengurangi pengeluaran",
    "target tabungan",
    "rekomendasi keuangan",
    "pengertian literasi keuangan",
]

RESPONSE_DATABASE = {
    "tips menabung": (
        "Tips Menabung yang Efektif:\n"
        "1. Terapkan metode 50/30/20 — 50% kebutuhan, 30% keinginan, 20% tabungan.\n"
        "2. Simpan di awal bulan sebelum digunakan (pay yourself first).\n"
        "3. Buat rekening tabungan terpisah dari rekening harian.\n"
        "4. Mulai dari nominal kecil tapi konsisten — Rp 50.000/hari = Rp 1,5 juta/bulan.\n"
        "5. Gunakan fitur Target Tabungan di FinSmart untuk pantau progres kamu."
    ),
    "cara mengatur anggaran": (
        "Cara Mengatur Anggaran Bulanan:\n"
        "1. Catat semua pemasukan bulanan terlebih dahulu.\n"
        "2. Bagi pengeluaran ke kategori: Makanan, Transport, Tagihan, Hiburan, dll.\n"
        "3. Tentukan batas maksimal untuk setiap kategori.\n"
        "4. Pantau pengeluaran harian agar tidak melebihi batas.\n"
        "5. FinSmart dapat mengingatkan kamu saat mendekati batas anggaran."
    ),
    "kategori pengeluaran": (
        "Kategori Pengeluaran di FinSmart:\n"
        "FinSmart mengklasifikasikan transaksi ke 10 kategori:\n"
        "1. Food (Makanan)\n"
        "2. Grocery (Belanja Kebutuhan)\n"
        "3. Transport (Transportasi)\n"
        "4. Entertainment (Hiburan)\n"
        "5. Healthcare (Kesehatan)\n"
        "6. Electronics (Elektronik)\n"
        "7. Online Shopping\n"
        "8. Travel (Perjalanan)\n"
        "9. Clothing (Pakaian)\n"
        "10. Bills (Tagihan)\n\n"
        "Kategori ditentukan otomatis oleh AI saat kamu input transaksi."
    ),
    "investasi untuk pemula": (
        "Panduan Investasi untuk Pemula:\n"
        "Sebelum investasi, pastikan sudah:\n"
        "- Punya dana darurat (3-6x pengeluaran bulanan)\n"
        "- Tidak punya hutang konsumtif\n"
        "- Tabungan rutin sudah berjalan\n\n"
        "Pilihan investasi untuk pemula:\n"
        "- Reksa Dana Pasar Uang — risiko rendah, cocok untuk pemula\n"
        "- Deposito — aman, bunga pasti\n"
        "- Obligasi Negara (ORI/SBR) — dijamin pemerintah\n\n"
        "Cek fitur Kesiapan Investasi di FinSmart untuk analisis profil keuanganmu."
    ),
    "cara mengurangi pengeluaran": (
        "Tips Mengurangi Pengeluaran:\n"
        "1. Identifikasi kategori pengeluaran terbesar dari dashboard FinSmart.\n"
        "2. Bedakan kebutuhan (harus) vs keinginan (bisa ditunda).\n"
        "3. Kurangi frekuensi makan di luar — masak sendiri bisa hemat 60%.\n"
        "4. Manfaatkan promo dan cashback saat belanja online.\n"
        "5. Batalkan langganan yang jarang dipakai (streaming, app, dll)."
    ),
    "target tabungan": (
        "Cara Menetapkan Target Tabungan:\n"
        "1. Tentukan tujuan spesifik: liburan, DP rumah, dana darurat, dll.\n"
        "2. Hitung nominal yang dibutuhkan dan tenggat waktunya.\n"
        "3. Bagi dengan jumlah bulan untuk mendapatkan target bulanan.\n"
        "4. Set target di fitur Sistem Anggaran FinSmart.\n"
        "5. FinSmart akan memberi notifikasi jika kamu on-track atau perlu penyesuaian."
    ),
    "rekomendasi keuangan": (
        "Rekomendasi Keuangan Personal:\n"
        "Berdasarkan data keuanganmu, FinSmart memberikan rekomendasi:\n"
        "- Analisis pola pengeluaran bulanan\n"
        "- Kategori mana yang paling banyak menguras anggaran\n"
        "- Prediksi pengeluaran bulan depan\n"
        "- Rekomendasi batas anggaran per kategori\n\n"
        "Masuk ke menu Analisis dan Rekomendasi AI untuk melihat insight personalmu."
    ),
    "pengertian literasi keuangan": (
        "Apa itu Literasi Keuangan?\n"
        "Literasi keuangan adalah kemampuan memahami dan menggunakan\n"
        "berbagai konsep keuangan secara efektif, meliputi:\n"
        "- Mengelola pendapatan dan pengeluaran\n"
        "- Merencanakan tabungan dan investasi\n"
        "- Memahami produk keuangan (kredit, asuransi, investasi)\n"
        "- Membuat keputusan keuangan yang bijak\n\n"
        "Data OJK 2024: literasi keuangan generasi muda Indonesia masih di bawah 50%."
    ),
}

RESPONSE_DEFAULT = (
    "Maaf, saya belum bisa menjawab pertanyaan tersebut.\n"
    "Kamu bisa tanya tentang:\n"
    "- Tips menabung\n"
    "- Cara mengatur anggaran\n"
    "- Kategori pengeluaran\n"
    "- Investasi untuk pemula\n"
    "- Cara mengurangi pengeluaran\n"
    "- Target tabungan"
)

print(f"Database FinBot siap: {len(RESPONSE_DATABASE)} intent terdaftar")

Database FinBot siap: 8 intent terdaftar


---
## 4. Kelas FinBot

Kelas `FinBot` mengelola alur deteksi intent dan pengembalian respons. Setiap percakapan disimpan dalam `chat_history` untuk keperluan logging.

In [6]:
class FinBot:
    """
    Chatbot keuangan FinSmart berbasis Hugging Face Zero-Shot Classification.

    Alur kerja:
    1. Terima pertanyaan pengguna
    2. Deteksi intent menggunakan Zero-Shot Classification
    3. Kembalikan respons sesuai intent yang terdeteksi
    """

    def __init__(self, classifier, intent_labels, response_db, threshold=0.10):
        self.classifier    = classifier
        self.intent_labels = intent_labels
        self.response_db   = response_db
        self.threshold     = threshold
        self.chat_history  = []

    def detect_intent(self, pertanyaan: str) -> tuple:
        """Deteksi intent dari pertanyaan menggunakan Zero-Shot Classification."""
        result     = self.classifier(
            pertanyaan,
            candidate_labels=self.intent_labels,
            hypothesis_template="Pertanyaan ini tentang {}."
        )
        top_intent = result["labels"][0]
        top_score  = result["scores"][0]
        return top_intent, top_score

    def chat(self, pertanyaan: str) -> dict:
        """Proses satu pertanyaan pengguna dan kembalikan respons."""
        pertanyaan = pertanyaan.strip()
        if not pertanyaan:
            return {"error": "Pertanyaan tidak boleh kosong."}

        intent, confidence = self.detect_intent(pertanyaan)

        if confidence >= self.threshold:
            respons = self.response_db.get(intent, RESPONSE_DEFAULT)
        else:
            respons = RESPONSE_DEFAULT
            intent  = "tidak_terdeteksi"

        entry = {
            "pertanyaan" : pertanyaan,
            "intent"     : intent,
            "confidence" : round(confidence * 100, 2),
            "respons"    : respons,
        }
        self.chat_history.append(entry)
        return entry

    def tampilkan_respons(self, result: dict):
        """Tampilkan hasil percakapan dengan format yang rapi."""
        print("=" * 60)
        print(f"Pertanyaan : {result['pertanyaan']}")
        print(f"Intent     : {result['intent']} ({result['confidence']}%)")
        print("-" * 60)
        print(f"FinBot     :\n{result['respons']}")
        print("=" * 60)


finbot = FinBot(
    classifier    = finbot_classifier,
    intent_labels = INTENT_LABELS,
    response_db   = RESPONSE_DATABASE,
    threshold     = 0.10
)

print("FinBot berhasil diinisialisasi")

FinBot berhasil diinisialisasi


---
## 5. Uji FinBot

In [7]:
pertanyaan_uji = [
    "Gimana cara biar bisa nabung lebih banyak tiap bulan?",
    "Aku mau mulai investasi tapi masih pemula, harus mulai dari mana?",
    "Pengeluaran makanku terlalu besar, gimana cara nguranginnya?",
    "Apa itu kategori pengeluaran di FinSmart?",
    "Bantu aku bikin target tabungan untuk liburan",
]

for pertanyaan in pertanyaan_uji:
    result = finbot.chat(pertanyaan)
    finbot.tampilkan_respons(result)
    print()

Pertanyaan : Gimana cara biar bisa nabung lebih banyak tiap bulan?
Intent     : rekomendasi keuangan (15.88%)
------------------------------------------------------------
FinBot     :
Rekomendasi Keuangan Personal:
Berdasarkan data keuanganmu, FinSmart memberikan rekomendasi:
- Analisis pola pengeluaran bulanan
- Kategori mana yang paling banyak menguras anggaran
- Prediksi pengeluaran bulan depan
- Rekomendasi batas anggaran per kategori

Masuk ke menu Analisis dan Rekomendasi AI untuk melihat insight personalmu.

Pertanyaan : Aku mau mulai investasi tapi masih pemula, harus mulai dari mana?
Intent     : investasi untuk pemula (23.45%)
------------------------------------------------------------
FinBot     :
Panduan Investasi untuk Pemula:
Sebelum investasi, pastikan sudah:
- Punya dana darurat (3-6x pengeluaran bulanan)
- Tidak punya hutang konsumtif
- Tabungan rutin sudah berjalan

Pilihan investasi untuk pemula:
- Reksa Dana Pasar Uang — risiko rendah, cocok untuk pemula
- Deposito

In [8]:
with open("finbot_chat_history.json", "w", encoding="utf-8") as f:
    json.dump(finbot.chat_history, f, ensure_ascii=False, indent=2)

print(f"Chat history disimpan   : finbot_chat_history.json")
print(f"Total percakapan        : {len(finbot.chat_history)}")

Chat history disimpan   : finbot_chat_history.json
Total percakapan        : 5


---
## 6. Sistem Rekomendasi Anggaran Personal

Kelas `RekomendasiAnggaran` menganalisis profil keuangan pengguna dan menghasilkan:

- Rekomendasi batas anggaran per kategori berdasarkan metode 50/30/20 dan standar OJK
- Deteksi kategori yang melebihi batas wajar
- Skor kesiapan investasi (0-100)
- Rekomendasi instrumen investasi sesuai profil risiko

In [9]:
class RekomendasiAnggaran:
    """
    Sistem rekomendasi anggaran personal FinSmart.

    Input  : profil keuangan pengguna (income, tabungan, pengeluaran, usia)
    Output : rekomendasi anggaran, skor investasi, instrumen yang sesuai
    """

    BOBOT_IDEAL = {
        "Food"           : 0.15,
        "Grocery"        : 0.10,
        "Transport"      : 0.10,
        "Bills"          : 0.10,
        "Healthcare"     : 0.05,
        "Entertainment"  : 0.05,
        "Online Shopping": 0.05,
        "Clothing"       : 0.05,
        "Electronics"    : 0.05,
        "Travel"         : 0.05,
    }

    INSTRUMEN_INVESTASI = {
        "konservatif": [
            "Tabungan berjangka / Deposito",
            "Obligasi Negara Ritel (ORI / SBR)",
            "Reksa Dana Pasar Uang",
        ],
        "moderat": [
            "Reksa Dana Pasar Uang",
            "Reksa Dana Pendapatan Tetap",
            "Obligasi Negara Ritel (ORI / SBR)",
            "P2P Lending terpercaya (OJK)",
        ],
        "agresif": [
            "Reksa Dana Saham",
            "Saham blue-chip (LQ45)",
            "ETF / Index Fund",
            "REITs (properti digital)",
        ],
    }

    def __init__(self, income_bulanan: float, tabungan_bulanan: float,
                 pengeluaran_per_kategori: dict, usia: int = 25):
        """
        Args:
            income_bulanan           : Total pemasukan per bulan (Rp)
            tabungan_bulanan         : Jumlah yang berhasil ditabung per bulan (Rp)
            pengeluaran_per_kategori : dict {kategori: total_pengeluaran_bulan_ini}
            usia                     : Usia pengguna untuk menentukan profil risiko
        """
        self.income            = income_bulanan
        self.tabungan          = tabungan_bulanan
        self.pengeluaran       = pengeluaran_per_kategori
        self.usia              = usia
        self.total_pengeluaran = sum(pengeluaran_per_kategori.values())
        self.rasio_tabungan    = tabungan_bulanan / income_bulanan if income_bulanan > 0 else 0

    def hitung_rekomendasi_anggaran(self) -> dict:
        """Hitung batas anggaran ideal per kategori dan bandingkan dengan pengeluaran aktual."""
        rekomendasi = {}
        for kategori, bobot in self.BOBOT_IDEAL.items():
            batas_ideal = self.income * bobot
            aktual      = self.pengeluaran.get(kategori, 0)
            selisih     = aktual - batas_ideal
            rekomendasi[kategori] = {
                "batas_ideal"   : round(batas_ideal),
                "aktual"        : round(aktual),
                "selisih"       : round(selisih),
                "status"        : "Over" if selisih > 0 else "Aman",
                "persen_income" : round((aktual / self.income) * 100, 1) if self.income > 0 else 0,
            }
        return rekomendasi

    def hitung_skor_investasi(self) -> dict:
        """
        Hitung skor kesiapan investasi (0-100) berdasarkan 4 faktor:
        - Rasio tabungan terhadap income    (max 40 poin)
        - Sisa income setelah pengeluaran   (max 30 poin)
        - Efisiensi rasio pengeluaran       (max 20 poin)
        - Usia pengguna                     (max 10 poin)
        """
        skor   = 0
        detail = {}

        skor_tabungan = min(40, int(self.rasio_tabungan / 0.20 * 40))
        skor += skor_tabungan
        detail["skor_tabungan"] = skor_tabungan

        sisa_income = self.income - self.total_pengeluaran - self.tabungan
        rasio_sisa  = max(0, sisa_income / self.income) if self.income > 0 else 0
        skor_sisa   = min(30, int(rasio_sisa * 100))
        skor += skor_sisa
        detail["skor_sisa_income"] = skor_sisa

        rasio_pengeluaran = self.total_pengeluaran / self.income if self.income > 0 else 1
        if rasio_pengeluaran <= 0.50:
            skor_efisiensi = 20
        elif rasio_pengeluaran <= 0.70:
            skor_efisiensi = 10
        else:
            skor_efisiensi = 0
        skor += skor_efisiensi
        detail["skor_efisiensi"] = skor_efisiensi

        if self.usia <= 25:
            skor_usia = 10
        elif self.usia <= 35:
            skor_usia = 7
        else:
            skor_usia = 5
        skor += skor_usia
        detail["skor_usia"] = skor_usia

        if skor >= 85:
            label         = "Siap Investasi"
            profil_risiko = "agresif"
        elif skor >= 70:
            label         = "Siap Investasi"
            profil_risiko = "moderat"
        elif skor >= 45:
            label         = "Hampir Siap — Perlu Sedikit Perbaikan"
            profil_risiko = "konservatif"
        else:
            label         = "Belum Siap — Fokus Perbaiki Keuangan Dulu"
            profil_risiko = "konservatif"

        return {
            "skor"         : skor,
            "label"        : label,
            "profil_risiko": profil_risiko,
            "detail_skor"  : detail,
        }

    def generate_rekomendasi(self) -> dict:
        """Generate laporan rekomendasi lengkap."""
        anggaran    = self.hitung_rekomendasi_anggaran()
        investasi   = self.hitung_skor_investasi()
        over_budget = [k for k, v in anggaran.items() if v["status"] == "Over"]
        instrumen   = self.INSTRUMEN_INVESTASI.get(investasi["profil_risiko"], [])

        return {
            "ringkasan": {
                "income_bulanan"     : self.income,
                "total_pengeluaran"  : self.total_pengeluaran,
                "tabungan_bulanan"   : self.tabungan,
                "rasio_tabungan_pct" : round(self.rasio_tabungan * 100, 1),
            },
            "rekomendasi_anggaran" : anggaran,
            "kategori_over_budget" : over_budget,
            "kesiapan_investasi"   : investasi,
            "rekomendasi_instrumen": instrumen,
        }

    def tampilkan_laporan(self):
        """Tampilkan laporan rekomendasi dengan format tabel yang rapi."""
        hasil = self.generate_rekomendasi()

        print("=" * 65)
        print("LAPORAN REKOMENDASI ANGGARAN — FinSmart")
        print("=" * 65)

        r = hasil["ringkasan"]
        print("\nRINGKASAN KEUANGAN:")
        print(f"  Income Bulanan    : Rp {r['income_bulanan']:>12,.0f}")
        print(f"  Total Pengeluaran : Rp {r['total_pengeluaran']:>12,.0f}")
        print(f"  Tabungan Bulanan  : Rp {r['tabungan_bulanan']:>12,.0f}")
        print(f"  Rasio Tabungan    : {r['rasio_tabungan_pct']}%")

        print("\nANGGARAN PER KATEGORI:")
        print(f"  {'Kategori':<18} {'Ideal':>12} {'Aktual':>12} {'Selisih':>12} Status")
        print(f"  {'-'*18} {'-'*12} {'-'*12} {'-'*12} {'-'*10}")
        for kat, v in hasil["rekomendasi_anggaran"].items():
            print(
                f"  {kat:<18} "
                f"Rp {v['batas_ideal']:>9,.0f} "
                f"Rp {v['aktual']:>9,.0f} "
                f"Rp {v['selisih']:>+9,.0f} "
                f"{v['status']}"
            )

        over = hasil["kategori_over_budget"]
        if over:
            print(f"\nKATEGORI OVER BUDGET : {', '.join(over)}")
            print("  Pertimbangkan untuk mengurangi pengeluaran di kategori ini.")

        inv = hasil["kesiapan_investasi"]
        print("\nKESIAPAN INVESTASI:")
        print(f"  Skor          : {inv['skor']}/100")
        print(f"  Status        : {inv['label']}")
        print(f"  Profil Risiko : {inv['profil_risiko'].capitalize()}")

        print("\nREKOMENDASI INSTRUMEN INVESTASI:")
        for instrumen in hasil["rekomendasi_instrumen"]:
            print(f"  - {instrumen}")

        print("=" * 65)
        return hasil


print("Kelas RekomendasiAnggaran berhasil didefinisikan")

Kelas RekomendasiAnggaran berhasil didefinisikan


---
## 7. Uji Sistem Rekomendasi Anggaran

In [10]:
print("=" * 65)
print("UJI KASUS 1: Pengguna A — Karyawan, 24 tahun")
print("=" * 65)

rekomendasi_1 = RekomendasiAnggaran(
    income_bulanan=5_000_000,
    tabungan_bulanan=1_000_000,
    pengeluaran_per_kategori={
        "Food"           : 800_000,
        "Grocery"        : 400_000,
        "Transport"      : 500_000,
        "Bills"          : 300_000,
        "Entertainment"  : 400_000,
        "Online Shopping": 300_000,
        "Healthcare"     : 100_000,
        "Clothing"       : 200_000,
        "Electronics"    : 0,
        "Travel"         : 0,
    },
    usia=24
)
hasil_1 = rekomendasi_1.tampilkan_laporan()

UJI KASUS 1: Pengguna A — Karyawan, 24 tahun
LAPORAN REKOMENDASI ANGGARAN — FinSmart

RINGKASAN KEUANGAN:
  Income Bulanan    : Rp    5,000,000
  Total Pengeluaran : Rp    3,000,000
  Tabungan Bulanan  : Rp    1,000,000
  Rasio Tabungan    : 20.0%

ANGGARAN PER KATEGORI:
  Kategori                  Ideal       Aktual      Selisih Status
  ------------------ ------------ ------------ ------------ ----------
  Food               Rp   750,000 Rp   800,000 Rp   +50,000 Over
  Grocery            Rp   500,000 Rp   400,000 Rp  -100,000 Aman
  Transport          Rp   500,000 Rp   500,000 Rp        +0 Aman
  Bills              Rp   500,000 Rp   300,000 Rp  -200,000 Aman
  Healthcare         Rp   250,000 Rp   100,000 Rp  -150,000 Aman
  Entertainment      Rp   250,000 Rp   400,000 Rp  +150,000 Over
  Online Shopping    Rp   250,000 Rp   300,000 Rp   +50,000 Over
  Clothing           Rp   250,000 Rp   200,000 Rp   -50,000 Aman
  Electronics        Rp   250,000 Rp         0 Rp  -250,000 Aman
  Tra

In [11]:
print("=" * 65)
print("UJI KASUS 2: Pengguna B — Mahasiswa, 21 tahun")
print("=" * 65)

rekomendasi_2 = RekomendasiAnggaran(
    income_bulanan=2_000_000,
    tabungan_bulanan=100_000,
    pengeluaran_per_kategori={
        "Food"           : 700_000,
        "Grocery"        : 200_000,
        "Transport"      : 300_000,
        "Bills"          : 200_000,
        "Entertainment"  : 300_000,
        "Online Shopping": 250_000,
        "Healthcare"     : 50_000,
        "Clothing"       : 150_000,
        "Electronics"    : 0,
        "Travel"         : 0,
    },
    usia=21
)
hasil_2 = rekomendasi_2.tampilkan_laporan()

UJI KASUS 2: Pengguna B — Mahasiswa, 21 tahun
LAPORAN REKOMENDASI ANGGARAN — FinSmart

RINGKASAN KEUANGAN:
  Income Bulanan    : Rp    2,000,000
  Total Pengeluaran : Rp    2,150,000
  Tabungan Bulanan  : Rp      100,000
  Rasio Tabungan    : 5.0%

ANGGARAN PER KATEGORI:
  Kategori                  Ideal       Aktual      Selisih Status
  ------------------ ------------ ------------ ------------ ----------
  Food               Rp   300,000 Rp   700,000 Rp  +400,000 Over
  Grocery            Rp   200,000 Rp   200,000 Rp        +0 Aman
  Transport          Rp   200,000 Rp   300,000 Rp  +100,000 Over
  Bills              Rp   200,000 Rp   200,000 Rp        +0 Aman
  Healthcare         Rp   100,000 Rp    50,000 Rp   -50,000 Aman
  Entertainment      Rp   100,000 Rp   300,000 Rp  +200,000 Over
  Online Shopping    Rp   100,000 Rp   250,000 Rp  +150,000 Over
  Clothing           Rp   100,000 Rp   150,000 Rp   +50,000 Over
  Electronics        Rp   100,000 Rp         0 Rp  -100,000 Aman
  Tra

In [12]:
with open("contoh_rekomendasi.json", "w", encoding="utf-8") as f:
    json.dump({
        "pengguna_A": hasil_1,
        "pengguna_B": hasil_2,
    }, f, ensure_ascii=False, indent=2)

print("Contoh rekomendasi disimpan : contoh_rekomendasi.json")

Contoh rekomendasi disimpan : contoh_rekomendasi.json


---
## 8. Dokumentasi Arsitektur Pipeline AI End-to-End

Seluruh komponen AI didokumentasikan dalam satu file JSON untuk keperluan pelaporan dan referensi tim back-end.

In [13]:
arsitektur = {
    "nama_proyek" : "FinSmart AI Pipeline",
    "versi"       : "1.0.0",
    "ai_engineer" : "Muhammad Syaiful",
    "capstone_id" : "CC26-PSU407",

    "komponen": {

        "1_klasifikasi_pengeluaran": {
            "deskripsi"    : "Model Deep Learning untuk mengklasifikasikan kategori pengeluaran",
            "model"        : "TensorFlow Functional API + XGBoost",
            "input"        : "Amount, PaymentMethod, Location, AccountType, TransactionType, "
                             "DeviceUsed, MerchantType, LoyaltyProgram, Weekday, Month, TimeOfDay, MerchantName",
            "output"       : "10 kelas kategori pengeluaran",
            "custom"       : "FinSmartCallback (Custom Callback — Main Quest)",
            "format_model" : "finsmart_model.keras",
            "endpoint_api" : "POST /classify",
        },

        "2_finbot_chatbot": {
            "deskripsi"    : "Chatbot keuangan berbasis NLP untuk menjawab pertanyaan pengguna",
            "model"        : "Hugging Face — facebook/bart-large-mnli",
            "teknik"       : "Zero-Shot Classification + Intent-Response Mapping",
            "input"        : "Pertanyaan pengguna dalam bahasa Indonesia",
            "output"       : "Respons keuangan berdasarkan intent yang terdeteksi",
            "jumlah_intent": len(INTENT_LABELS),
            "endpoint_api" : "POST /finbot/chat",
        },

        "3_rekomendasi_anggaran": {
            "deskripsi"    : "Sistem rekomendasi anggaran personal berbasis profil keuangan",
            "metode"       : "Rule-Based Scoring + Threshold Analysis",
            "input"        : "income_bulanan, tabungan_bulanan, pengeluaran_per_kategori, usia",
            "output"       : "Rekomendasi anggaran per kategori + skor kesiapan investasi",
            "acuan"        : "Metode 50/30/20 + Standar Literasi Keuangan OJK",
            "endpoint_api" : "POST /rekomendasi",
        },

        "4_api_serving": {
            "deskripsi"  : "REST API untuk menyajikan semua komponen AI ke front-end",
            "framework"  : "FastAPI + Uvicorn",
            "endpoints"  : [
                "GET  /              — health check",
                "POST /classify      — klasifikasi pengeluaran",
                "POST /finbot/chat   — chatbot FinBot",
                "POST /rekomendasi   — rekomendasi anggaran",
                "GET  /model-info    — info model",
                "GET  /valid-values  — nilai valid tiap fitur",
                "GET  /docs          — Swagger UI",
            ],
            "integrasi"  : "Dipanggil oleh back-end Node.js/Express via HTTP",
        },
    },

    "alur_data": [
        "1. Pengguna input transaksi di front-end (React.js)",
        "2. Front-end kirim request ke back-end (Node.js/Express)",
        "3. Back-end forward ke AI Service (FastAPI) via HTTP POST",
        "4. FastAPI load model, lakukan preprocessing, jalankan inference",
        "5. Hasil prediksi dikembalikan ke back-end lalu ke front-end",
        "6. Dashboard menampilkan kategori dan insight kepada pengguna",
    ],

    "file_artifacts": [
        "finsmart_model.keras      — Model TF Functional API",
        "xgb_model.pkl             — Model XGBoost final",
        "encoders.pkl              — Label encoders semua fitur",
        "scaler.pkl                — StandardScaler",
        "model_metadata.json       — Metadata dan performa model",
        "training_log.json         — Log training per epoch",
        "finbot_chat_history.json  — Contoh percakapan FinBot",
        "contoh_rekomendasi.json   — Contoh output rekomendasi",
        "arsitektur_ai.json        — Dokumentasi arsitektur pipeline",
        "main_complete.py          — FastAPI server lengkap",
    ],
}

with open("arsitektur_ai.json", "w", encoding="utf-8") as f:
    json.dump(arsitektur, f, ensure_ascii=False, indent=2)

print("Dokumentasi arsitektur disimpan : arsitektur_ai.json")
print()
print("ALUR PIPELINE AI END-TO-END:")
for step in arsitektur["alur_data"]:
    print(f"  {step}")
print()
print("KOMPONEN AI:")
for nama, detail in arsitektur["komponen"].items():
    model_info = detail.get("model", detail.get("metode", detail.get("framework", "-")))
    endpoint   = detail.get("endpoint_api", detail.get("endpoints", ["-"])[0])
    print(f"  [{nama}]")
    print(f"    Deskripsi    : {detail['deskripsi']}")
    print(f"    Model/Metode : {model_info}")
    print(f"    Endpoint API : {endpoint}")
    print()

Dokumentasi arsitektur disimpan : arsitektur_ai.json

ALUR PIPELINE AI END-TO-END:
  1. Pengguna input transaksi di front-end (React.js)
  2. Front-end kirim request ke back-end (Node.js/Express)
  3. Back-end forward ke AI Service (FastAPI) via HTTP POST
  4. FastAPI load model, lakukan preprocessing, jalankan inference
  5. Hasil prediksi dikembalikan ke back-end lalu ke front-end
  6. Dashboard menampilkan kategori dan insight kepada pengguna

KOMPONEN AI:
  [1_klasifikasi_pengeluaran]
    Deskripsi    : Model Deep Learning untuk mengklasifikasikan kategori pengeluaran
    Model/Metode : TensorFlow Functional API + XGBoost
    Endpoint API : POST /classify

  [2_finbot_chatbot]
    Deskripsi    : Chatbot keuangan berbasis NLP untuk menjawab pertanyaan pengguna
    Model/Metode : Hugging Face — facebook/bart-large-mnli
    Endpoint API : POST /finbot/chat

  [3_rekomendasi_anggaran]
    Deskripsi    : Sistem rekomendasi anggaran personal berbasis profil keuangan
    Model/Metode : Ru

---
## 9. FastAPI Lengkap — Semua Endpoint

File `main_complete.py` adalah REST API lengkap yang mencakup tiga endpoint utama:

- `POST /classify` — klasifikasi kategori pengeluaran
- `POST /finbot/chat` — chatbot FinBot
- `POST /rekomendasi` — rekomendasi anggaran personal

Cara menjalankan:
```
uvicorn main_complete:app --reload --port 8000
```
Dokumentasi otomatis: `http://localhost:8000/docs`

In [14]:
main_complete = '''
import numpy as np
import pickle
import json
import joblib
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from transformers import pipeline as hf_pipeline

app = FastAPI(
    title="FinSmart AI Service",
    description="API lengkap untuk klasifikasi pengeluaran, FinBot, dan rekomendasi anggaran",
    version="1.0.0"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"]
)

model    = joblib.load("xgb_model.pkl")
encoders = pickle.load(open("encoders.pkl", "rb"))
scaler   = pickle.load(open("scaler.pkl",   "rb"))
metadata = json.load(open("model_metadata.json"))

finbot_clf = hf_pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

FITUR_KAT = [
    "PaymentMethod", "Location", "AccountType", "TransactionType",
    "DeviceUsed", "MerchantType", "LoyaltyProgram", "Weekday",
    "Month", "TimeOfDay", "MerchantName"
]

INTENT_LABELS = [
    "tips menabung", "cara mengatur anggaran", "kategori pengeluaran",
    "investasi untuk pemula", "cara mengurangi pengeluaran",
    "target tabungan", "rekomendasi keuangan", "pengertian literasi keuangan",
]

RESPONSE_DB = {
    "tips menabung"              : "Terapkan metode 50/30/20. Simpan di awal bulan sebelum digunakan.",
    "cara mengatur anggaran"     : "Catat pemasukan, bagi ke kategori, tentukan batas, dan pantau setiap hari.",
    "kategori pengeluaran"       : "FinSmart mengklasifikasikan ke 10 kategori: Food, Grocery, Transport, Entertainment, Healthcare, Electronics, Online Shopping, Travel, Clothing, Bills.",
    "investasi untuk pemula"     : "Pastikan punya dana darurat dulu, lalu mulai dari Reksa Dana Pasar Uang atau Deposito.",
    "cara mengurangi pengeluaran": "Identifikasi kategori terbesar, bedakan kebutuhan vs keinginan, dan pantau tren pengeluaran.",
    "target tabungan"            : "Tentukan tujuan spesifik, hitung nominal dan tenggat waktu, bagi per bulan, dan set target di FinSmart.",
    "rekomendasi keuangan"       : "Lihat menu Analisis dan Rekomendasi AI di FinSmart untuk insight personal.",
    "pengertian literasi keuangan": "Kemampuan memahami dan menggunakan konsep keuangan: mengelola income, menabung, investasi, dan membuat keputusan finansial yang bijak.",
}
RESPONSE_DEFAULT = "Maaf, saya belum bisa menjawab itu. Coba tanya tentang tips menabung, anggaran, atau investasi."

BOBOT_IDEAL = {
    "Food": 0.15, "Grocery": 0.10, "Transport": 0.10, "Bills": 0.10,
    "Healthcare": 0.05, "Entertainment": 0.05, "Online Shopping": 0.05,
    "Clothing": 0.05, "Electronics": 0.05, "Travel": 0.05,
}

INSTRUMEN = {
    "konservatif": ["Tabungan berjangka / Deposito", "Obligasi Negara (ORI/SBR)", "Reksa Dana Pasar Uang"],
    "moderat"    : ["Reksa Dana Pendapatan Tetap", "ORI/SBR", "P2P Lending OJK"],
    "agresif"    : ["Reksa Dana Saham", "Saham blue-chip (LQ45)", "ETF / Index Fund"],
}


class TransaksiInput(BaseModel):
    Amount: float
    PaymentMethod: str
    Location: str
    AccountType: str
    TransactionType: str
    DeviceUsed: str
    MerchantType: str
    LoyaltyProgram: str
    Weekday: str
    Month: str
    TimeOfDay: str
    MerchantName: str

class ChatInput(BaseModel):
    pertanyaan: str

class RekomendasiInput(BaseModel):
    income_bulanan: float
    tabungan_bulanan: float
    usia: int = 25
    pengeluaran: dict


@app.get("/", tags=["Health"])
def root():
    return {"status": "ok", "service": "FinSmart AI Service", "version": "1.0.0"}

@app.post("/classify", tags=["Klasifikasi"])
def classify(transaksi: TransaksiInput):
    data = transaksi.dict()
    row  = {}
    for col in FITUR_KAT:
        le = encoders[col]
        if data[col] not in le.classes_:
            raise HTTPException(422, f"Nilai tidak valid: {col}={data[col]}")
        row[col] = le.transform([data[col]])[0]
    row["Amount"] = scaler.transform([[data["Amount"]]])[0][0]
    X     = np.array([[row[f] for f in FITUR_KAT + ["Amount"]]])
    proba = model.predict_proba(X)[0]
    idx   = int(np.argmax(proba))
    kategori = encoders["Category"].inverse_transform([idx])[0]
    semua    = encoders["Category"].classes_
    top3     = dict(sorted(
        {k: round(float(p), 4) for k, p in zip(semua, proba)}.items(),
        key=lambda x: x[1], reverse=True
    )[:3])
    return {"kategori": kategori, "confidence": round(float(proba[idx]) * 100, 2), "top3_prob": top3}

@app.post("/finbot/chat", tags=["FinBot"])
def finbot_chat(body: ChatInput):
    if not body.pertanyaan.strip():
        raise HTTPException(422, "Pertanyaan tidak boleh kosong.")
    result     = finbot_clf(
        body.pertanyaan,
        candidate_labels=INTENT_LABELS,
        hypothesis_template="Pertanyaan ini tentang {}."
    )
    intent     = result["labels"][0]
    confidence = result["scores"][0]
    respons    = RESPONSE_DB.get(intent, RESPONSE_DEFAULT) if confidence >= 0.30 else RESPONSE_DEFAULT
    return {"intent": intent, "confidence": round(confidence * 100, 2), "respons": respons}

@app.post("/rekomendasi", tags=["Rekomendasi"])
def rekomendasi_anggaran(body: RekomendasiInput):
    income   = body.income_bulanan
    anggaran = {}
    for kat, bobot in BOBOT_IDEAL.items():
        ideal  = income * bobot
        aktual = body.pengeluaran.get(kat, 0)
        anggaran[kat] = {
            "batas_ideal": round(ideal),
            "aktual"     : round(aktual),
            "selisih"    : round(aktual - ideal),
            "status"     : "Over" if aktual > ideal else "Aman",
        }
    total_exp = sum(body.pengeluaran.values())
    rasio_tab = body.tabungan_bulanan / income if income > 0 else 0
    skor      = min(40, int(rasio_tab / 0.20 * 40))
    sisa      = max(0, (income - total_exp - body.tabungan_bulanan) / income)
    skor     += min(30, int(sisa * 100))
    ratio_exp = total_exp / income if income > 0 else 1
    skor     += 20 if ratio_exp <= 0.50 else (10 if ratio_exp <= 0.70 else 0)
    skor     += 10 if body.usia <= 25 else (7 if body.usia <= 35 else 5)
    profil    = "agresif" if skor >= 85 else ("moderat" if skor >= 70 else "konservatif")
    label     = "Siap Investasi" if skor >= 70 else ("Hampir Siap" if skor >= 45 else "Belum Siap")
    return {
        "rekomendasi_anggaran" : anggaran,
        "kesiapan_investasi"   : {"skor": skor, "label": label, "profil_risiko": profil},
        "rekomendasi_instrumen": INSTRUMEN.get(profil, []),
        "over_budget"          : [k for k, v in anggaran.items() if v["status"] == "Over"],
    }

@app.get("/model-info", tags=["Info"])
def model_info():
    return metadata

@app.get("/valid-values", tags=["Info"])
def valid_values():
    return {col: list(encoders[col].classes_) for col in FITUR_KAT}
'''

with open("main_complete.py", "w", encoding="utf-8") as f:
    f.write(main_complete.strip())

print("FastAPI lengkap digenerate : main_complete.py")
print()
print("Cara menjalankan:")
print("  uvicorn main_complete:app --reload --port 8000")
print()
print("Endpoint yang tersedia:")
print("  GET  /              -> health check")
print("  POST /classify      -> klasifikasi pengeluaran")
print("  POST /finbot/chat   -> chatbot FinBot")
print("  POST /rekomendasi   -> rekomendasi anggaran")
print("  GET  /model-info    -> informasi model")
print("  GET  /valid-values  -> nilai valid tiap fitur")
print("  GET  /docs          -> Swagger UI")

FastAPI lengkap digenerate : main_complete.py

Cara menjalankan:
  uvicorn main_complete:app --reload --port 8000

Endpoint yang tersedia:
  GET  /              -> health check
  POST /classify      -> klasifikasi pengeluaran
  POST /finbot/chat   -> chatbot FinBot
  POST /rekomendasi   -> rekomendasi anggaran
  GET  /model-info    -> informasi model
  GET  /valid-values  -> nilai valid tiap fitur
  GET  /docs          -> Swagger UI


---
## 10. Ringkasan

In [15]:
print("=" * 65)
print("RINGKASAN TUGAS AI ENGINEER — FinSmart CC26-PSU407")
print("=" * 65)

print("\nMAIN QUEST (Notebook FinSmart_AI_Engineer_Final.ipynb):")
print("  1. Model TensorFlow Functional API   build_finsmart_model()")
print("  2. Custom Callback                   FinSmartCallback")
print("  3. Model disimpan format .keras       finsmart_model.keras")
print("  4. Kode inference                    FinSmartInference")

print("\nSIDE QUEST:")
print("  1. REST API FastAPI                  main_complete.py")
print("  2. Target akurasi >= 85%             xgb_model.pkl (99.83%)")

print("\nTUGAS JOB DESK AI ENGINEER (Notebook ini):")
print("  1. Arsitektur pipeline AI end-to-end   arsitektur_ai.json")
print("  2. FinBot berbasis NLP (Hugging Face)  FinBot class")
print("  3. Rekomendasi anggaran personal       RekomendasiAnggaran class")
print("  4. Integrasi semua komponen via API    main_complete.py")

print("\nFile yang dihasilkan:")
files = [
    ("finbot_chat_history.json" , "Log percakapan FinBot"),
    ("contoh_rekomendasi.json"  , "Contoh output rekomendasi anggaran"),
    ("arsitektur_ai.json"       , "Dokumentasi arsitektur pipeline AI"),
    ("main_complete.py"         , "FastAPI server dengan semua endpoint"),
]
for fname, desc in files:
    status = "[Ada]" if os.path.exists(fname) else "[Belum]"
    print(f"  {status} {fname:35s} : {desc}")

print("\n" + "=" * 65)

RINGKASAN TUGAS AI ENGINEER — FinSmart CC26-PSU407

MAIN QUEST (Notebook FinSmart_AI_Engineer_Final.ipynb):
  1. Model TensorFlow Functional API   build_finsmart_model()
  2. Custom Callback                   FinSmartCallback
  3. Model disimpan format .keras       finsmart_model.keras
  4. Kode inference                    FinSmartInference

SIDE QUEST:
  1. REST API FastAPI                  main_complete.py
  2. Target akurasi >= 85%             xgb_model.pkl (99.83%)

TUGAS JOB DESK AI ENGINEER (Notebook ini):
  1. Arsitektur pipeline AI end-to-end   arsitektur_ai.json
  2. FinBot berbasis NLP (Hugging Face)  FinBot class
  3. Rekomendasi anggaran personal       RekomendasiAnggaran class
  4. Integrasi semua komponen via API    main_complete.py

File yang dihasilkan:
  [Ada] finbot_chat_history.json            : Log percakapan FinBot
  [Ada] contoh_rekomendasi.json             : Contoh output rekomendasi anggaran
  [Ada] arsitektur_ai.json                  : Dokumentasi arsitektur p